# Interactive Loss Function Explorer

Choose the **best use case first**, then read a detailed explanation of the mathematical terms and assumptions, run numerical examples, inspect **PyTorch autograd gradients**, and graph the penalty and its gradient.

This is a **standalone notebook**, with no dependencies on other repository files. Run all cells once to activate the widgets. Requires `numpy pandas matplotlib torch ipywidgets`.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Loss functions, definitions, assumptions, formulas and gradient implementations

In [ ]:
"""Nine losses, educational scalar parameterization and glossary."""
import numpy as np
import torch
import torch.nn.functional as F

LOSSES = {
'Mean Squared Error (MSE)': dict(case='Predict continuous quantities; make large mistakes particularly costly.', terms='error = prediction − target; square makes every error nonnegative; mean averages across samples.', assumptions='Numeric target and predictions; outliers carry disproportionately large influence; assumes squared error is a suitable task penalty; often linked to Gaussian noise likelihood.', caveats='A single extreme outlier can dominate. Report MAE too.', formula='mean((prediction − target)²)'),
'Mean Absolute Error (MAE)': dict(case='Regression with occasional large errors or a focus on typical absolute deviation.',terms='absolute value removes sign; mean averages all sample errors.',assumptions='Numeric target; linear rather than quadratic penalty; optimum constant prediction is a median; zero residual has a nondifferentiable kink (subgradient is used).',caveats='Weaker gradient signal than MSE for large errors.', formula='mean(|prediction − target|)'),
'Binary Cross-Entropy': dict(case='Binary classification with labels 0/1.',terms='p is estimated probability of positive class; y is 0 or 1; log heavily penalizes confident wrong predictions.',assumptions='Bernoulli outcomes; examples represented by logits or probabilities; use BCE-with-logits for raw scores and numerical stability; distinguish imbalance from accuracy.',caveats='Applying sigmoid twice is wrong; probability calibration must be checked.',formula='−mean(y log p + (1−y) log(1−p))'),
'Categorical Cross-Entropy': dict(case='Exactly one mutually exclusive class among three or more.',terms='logits are unconstrained scores; softmax converts logits to class probabilities; true class index selects one log-probability.',assumptions='One true class index per example; logits input for PyTorch cross_entropy; class IDs 0..K−1; mutually exclusive categories.',caveats='Do not softmax logits before torch.nn.functional.cross_entropy.',formula='−mean(log softmax(logits)[true_class])'),
'Hinge Loss': dict(case='Margin-based binary classifier such as a linear SVM.',terms='y is label −1 or +1; score is raw decision value; margin is y×score; desired margin ≥ 1.',assumptions='Binary signed labels −1/+1, not 0/1; raw margin scores rather than probabilities; margin scale depends on feature and weight regularization.',caveats='Not a calibrated probability loss; plot plateaus after achieving the margin.',formula='mean(max(0, 1 − y×score))'),
'Log-Cosh Loss': dict(case='Smooth robust-ish regression with less extreme outlier sensitivity than MSE.',terms='cosh is hyperbolic cosine; log(cosh(error)) ≈ error²/2 near zero and |error|−log2 for large error.',assumptions='Real-valued residuals; smooth differentiable objective; use numerically stable implementation for large inputs.',caveats='Units and objective scaling differ from MSE/MAE.',formula='mean(log(cosh(prediction − target)))'),
'Huber Loss': dict(case='Regression requiring quadratic small errors and linear large-error penalty.',terms='delta controls transition between quadratic and linear regimes; |error| ≤ delta is quadratic.',assumptions='Numeric residuals; choose delta in target units; scaled features and targets affect meaningful delta.',caveats='PyTorch smooth_l1_loss has different scaling from huber_loss.',formula='0.5e² if |e|≤δ else δ(|e|−0.5δ)'),
'KL Divergence': dict(case='Match a student distribution to a reference/teacher distribution.',terms='P is reference probability distribution; Q is approximating distribution; sum P log(P/Q). Direction matters.',assumptions='P and Q nonnegative, normalized and over the same classes; Q should not assign zero probability where P>0; PyTorch kl_div takes log Q as input and P as target.',caveats='KL(P||Q) differs from KL(Q||P); nonnegative but not symmetric.',formula='Σ Pᵢ log(Pᵢ / Qᵢ)'),
'Cosine Similarity Loss': dict(case='Learn vector embeddings pointing in similar directions.',terms='dot product measures alignment; Euclidean norms normalize vector length; loss is 1 − cosine.',assumptions='Nonzero vectors; angular alignment matters more than magnitude; embeddings use same dimension.',caveats='Vectors with same direction but very different lengths score identically.',formula='1 − (u·v)/(||u|| ||v||)'),
}

def values(name, xs, delta=1.):
    """Return plot y-values and a PyTorch autograd gradient for adjustable scalar x."""
    x = torch.tensor(np.asarray(xs),dtype=torch.float64,requires_grad=True)
    z = torch.zeros_like(x)
    if name.startswith('Mean Squared'): loss=(x-z).square()
    elif name.startswith('Mean Absolute'): loss=(x-z).abs()
    elif name.startswith('Binary'): loss=F.binary_cross_entropy_with_logits(x,torch.ones_like(x),reduction='none')
    elif name.startswith('Categorical'): loss=F.cross_entropy(torch.stack([x,z,z],dim=1),torch.zeros(len(x),dtype=torch.long),reduction='none')
    elif name.startswith('Hinge'): loss=torch.relu(1-x)
    elif name.startswith('Log-Cosh'): loss=torch.logaddexp(x,-x)-np.log(2.)
    elif name.startswith('Huber'): loss=F.huber_loss(x,z,delta=delta,reduction='none')
    elif name.startswith('KL'):
        p=torch.tensor([0.8,0.2],dtype=x.dtype).repeat(len(x),1)
        q=torch.stack([torch.sigmoid(x),torch.sigmoid(-x)],dim=1)
        loss=(p*(p.log()-q.log())).sum(dim=1)
    else:
        u=torch.stack([torch.ones_like(x),z],dim=1)
        v=torch.stack([x,torch.ones_like(x)],dim=1)
        loss=1-F.cosine_similarity(u,v,dim=1)
    grad=torch.autograd.grad(loss.sum(),x)[0]
    return loss.detach().numpy(),grad.detach().numpy()


## Interactive loss-function menu

In [ ]:
import ipywidgets as widgets
from IPython.display import display,clear_output,Markdown
import matplotlib.pyplot as plt
import pandas as pd

choose=widgets.Dropdown(options=[(f"{x['case']}  →  {name}",name) for name,x in LOSSES.items()],description="Best use case:",style={'description_width':'initial'},layout=widgets.Layout(width='98%'))
delta=widgets.FloatSlider(value=1,min=.1,max=5,step=.1,description='Huber delta:',style={'description_width':'initial'})
explain_button=widgets.Button(description='Explain terms',button_style='info')
calculate_button=widgets.Button(description='Calculate + gradients',button_style='success')
visual_button=widgets.Button(description='Plot loss + gradient')
all_button=widgets.Button(description='Show all losses')
out=widgets.Output()

def explain_loss(*_):
    with out:
        clear_output(wait=True)
        entry=LOSSES[choose.value]
        display(Markdown(f"### {choose.value}\n**Best use case:** {entry['case']}  \n**Formula:** `{entry['formula']}`  \n**Terms explained:** {entry['terms']}  \n**Assumptions:** {entry['assumptions']}  \n**Common pitfall:** {entry['caveats']}"))
        display(Markdown('**Interpretation:** `loss` is the penalty minimized during training; `gradient` is how changing the scalar input changes the penalty. The x-axis parameter means a residual for regression, a logit for cross-entropy, a margin for hinge, a log-odds parameter for KL, or an embedding component for cosine. Therefore x-axis meanings differ between losses.'))

def compute_loss(make_plot=False):
    with out:
        clear_output(wait=True)
        entry=LOSSES[choose.value]
        display(Markdown(f"### {choose.value}\n**Assumptions:** {entry['assumptions']}"))
        xs=np.array([-3.,-1.,0.,1.,3.])
        ys,grad=values(choose.value,xs,delta.value)
        display(pd.DataFrame({'input':xs,'loss':ys,'PyTorch autograd dL/dx':grad}).round(5))
        if make_plot:
            xs=np.linspace(-5,5,401)
            ys,grad=values(choose.value,xs,delta.value)
            fig,ax=plt.subplots(figsize=(9,5))
            ax.plot(xs,ys,label='Loss'); ax.plot(xs,grad,label='Autograd gradient',alpha=.8)
            ax.axhline(0,color='gray',lw=.6);ax.set(xlabel='Scalar input (meaning depends on chosen loss)',ylabel='Value',title=choose.value)
            ax.legend();ax.grid(alpha=.2);plt.show()

def show_losses(*_):
    with out:
        clear_output(wait=True)
        display(pd.DataFrame([{'Loss':n,'Best use case':v['case'],'Formula':v['formula']} for n,v in LOSSES.items()]))
explain_button.on_click(explain_loss)
calculate_button.on_click(lambda _:compute_loss(False))
visual_button.on_click(lambda _:compute_loss(True))
all_button.on_click(show_losses)
display(widgets.VBox([choose,delta,widgets.HBox([explain_button,calculate_button,visual_button,all_button]),out]))
explain_loss()
